# Cài đặt lại FPClose bằng Julia — Notebook demo

**Đồ án môn Khai thác dữ liệu.** Cài đặt lại thuật toán **FPClose**, khai thác tập phổ biến đóng (closed frequent itemsets) bằng FP-tree, theo bài báo:

> G. Grahne và J. Zhu, "Fast Algorithms for Frequent Itemset Mining Using FP-Trees", *IEEE Transactions on Knowledge and Data Engineering*, 17(10), 2005. DOI: [10.1109/TKDE.2005.166](https://doi.org/10.1109/TKDE.2005.166)

**Thành viên nhóm:** TODO — Họ tên (MSSV), Họ tên (MSSV), ...

- Mã nguồn: https://github.com/DRAGON-1312/FPClose
- Dataset benchmark đã chuẩn hóa (Hugging Face): https://huggingface.co/datasets/hytrang/fpclose-fim-benchmarks

**Cách chạy:** mở trên Google Colab, chọn *Runtime → Run all*. Notebook dùng kernel **Python 3**; mọi bước Julia được gọi qua `!julia ...`. Có thể chạy lại nhiều lần: bước cài đặt, clone và tải dữ liệu tự bỏ qua nếu đã có. Mỗi mục tự kiểm tra điều kiện của mình (dữ liệu, Java, thư viện Python) và chỉ phụ thuộc mục 1, nên một mục lỗi không làm hỏng các mục khác.

Nội dung: 1. Cài đặt · 1b. Chuẩn bị dữ liệu · 2. Unit test · 3. Demo FPClose · 4. Đối chiếu SPMF · 5. Thực nghiệm · 6. Ứng dụng · 7. Kết luận

## 1. Cài đặt môi trường

Ô cấu hình bên dưới là chỗ duy nhất cần sửa. **Muốn chạy thử một nhánh khác trước khi merge thì đổi `BRANCH`.** `JULIA_VERSION` khớp `julia_version` trong `Manifest.toml` và phiên bản ghi trong `results/ENVIRONMENT.md`. Ô này cũng định nghĩa các hàm tiện ích (`sh`, `ensure_data`, `ensure_java`, `need_py`) mà các mục sau dùng để tự chuẩn bị điều kiện.

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL = 'https://github.com/DRAGON-1312/FPClose.git'
BRANCH = 'main'            # đổi thành nhánh khác (vd 'feature/fast-reader') để chạy thử trước khi merge
JULIA_VERSION = '1.13.4'   # khớp Manifest.toml (julia_version) và results/ENVIRONMENT.md
REPO_DIR = '/content/FPClose'
MACHINE_LABEL = 'colab'    # nhãn cột machine cho các số đo minh họa trên Colab

import os, glob, time, importlib
os.environ['FPCLOSE_MACHINE'] = MACHINE_LABEL
T_START = time.time()

# ---- Hàm tiện ích: mỗi mục sau tự kiểm tra/chuẩn bị điều kiện của mình bằng các hàm này ----
def sh(cmd):
    """Chạy lệnh shell (từ gốc repo nếu đã clone), output hiện ngay trong ô."""
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
    get_ipython().system(cmd)

def ensure_data(names):
    """Bảo đảm có data/benchmark/<name>.txt; thiếu thì gọi scripts/data_setup.jl --only <các file thiếu>."""
    path = lambda n: os.path.join(REPO_DIR, 'data', 'benchmark', f'{n}.txt')
    missing = [n for n in names if not os.path.exists(path(n))]
    if missing:
        print('Thiếu dữ liệu', missing, '-> chạy scripts/data_setup.jl --only', ','.join(missing))
        sh('julia --project=. scripts/data_setup.jl --only ' + ','.join(missing))
    ok = all(os.path.exists(path(n)) for n in names)
    print('Dữ liệu', names, 'sẵn sàng' if ok else 'VẪN THIẾU, bỏ qua bước cần dữ liệu này')
    return ok

def ensure_java():
    """Bảo đảm có Java 25 và tools/spmf.jar bằng setup_tools() có sẵn trong scripts/reference_check.jl
    (trên Linux tự tải Temurin 25), rồi thêm JDK vào PATH: compare_spmf.jl và run_benchmark_grid.jl gọi `java` trên PATH."""
    jdk_bins = lambda: sorted(glob.glob(os.path.join(REPO_DIR, 'tools', 'jdk-25*', 'bin')))
    if not jdk_bins() or not os.path.exists(os.path.join(REPO_DIR, 'tools', 'spmf.jar')):
        sh('''julia --project=. -e 'include("scripts/reference_check.jl"); setup_tools(TOOLS_DIR; with_fimi = false)' ''')
    bins = jdk_bins()
    if bins and bins[-1] not in os.environ['PATH'].split(os.pathsep):
        os.environ['PATH'] = bins[-1] + os.pathsep + os.environ['PATH']
    print('JDK:', bins[-1] if bins else 'không thấy tools/jdk-25*, dùng java có sẵn trên máy')
    sh('java -version 2>&1 | head -n 1')

def need_py(*pkgs):
    """Import được thì thôi, không thì pip install (Colab đã có sẵn pandas, IPython)."""
    for p in pkgs:
        try:
            importlib.import_module(p)
        except ImportError:
            sh(f'pip install -q {p}')

print('Nhánh:', BRANCH, '| Julia', JULIA_VERSION, '| thư mục', REPO_DIR)

In [ ]:
# Cài Julia từ tarball chính thức vào /usr (bỏ qua nếu `julia --version` đã đúng phiên bản).
import subprocess
def julia_version():
    try:
        return subprocess.run(['julia', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return ''

if julia_version() == f'julia version {JULIA_VERSION}':
    print('Đã có', julia_version(), '-> bỏ qua bước cài')
else:
    major_minor = '.'.join(JULIA_VERSION.split('.')[:2])
    url = f'https://julialang-s3.julialang.org/bin/linux/x64/{major_minor}/julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
    print('Tải', url)
    sh(f'curl -fsSL -o /tmp/julia.tar.gz {url} && tar -xzf /tmp/julia.tar.gz -C /usr --strip-components 1 && rm -f /tmp/julia.tar.gz')
!julia --version

In [ ]:
# Clone repo (bỏ qua nếu đã có) rồi chuyển sang nhánh BRANCH. Từ đây mọi lệnh chạy từ gốc repo.
![ -d {REPO_DIR}/.git ] || git clone --quiet {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin {BRANCH} && git -C {REPO_DIR} checkout --quiet -B {BRANCH} FETCH_HEAD
%cd {REPO_DIR}
!git log -1 --format='%h %s (%ci)'

In [ ]:
%%time
# Cài gói theo Manifest.toml (repo chỉ dùng thư viện chuẩn của Julia), biên dịch trước, in versioninfo.
!julia --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.precompile(); using FPClose, InteractiveUtils; versioninfo()'

## 1b. Chuẩn bị dữ liệu

`scripts/data_setup.jl` (chế độ mặc định) tải 9 dataset benchmark đã chuẩn hóa từ Hugging Face `hytrang/fpclose-fim-benchmarks` và kiểm SHA-256, sinh 5 tập con Retail (`data/scalability/`) và 6 CSDL tổng hợp theo độ dài giao dịch (`data/synthetic/`), rồi kiểm hash của các file sinh ra. Mọi cấu hình đọc từ `data/meta/*.csv`. Nếu đã đủ file thì bỏ qua. Các mục sau dùng dữ liệu này, nhưng vẫn tự kiểm tra và gọi lại `data_setup.jl --only ...` nếu thiếu.

In [ ]:
%%time
count_files = lambda pattern: len(glob.glob(os.path.join(REPO_DIR, pattern)))
n_bench = count_files('data/benchmark/*.txt')
n_scal = count_files('data/scalability/*.txt')
n_syn = count_files('data/synthetic/*.txt')
print(f'Hiện có: {n_bench}/9 benchmark, {n_scal}/5 tập con Retail, {n_syn}/6 synthetic')
if n_bench >= 9 and n_scal >= 5 and n_syn >= 6:
    print('Đủ dữ liệu -> bỏ qua data_setup.jl')
else:
    sh('julia --project=. scripts/data_setup.jl')

In [ ]:
need_py('pandas')
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
for p in ['data/meta/dataset_statistics.csv', 'data/meta/synthetic_statistics.csv']:
    full = os.path.join(REPO_DIR, p)
    print('==', p)
    display(pd.read_csv(full)) if os.path.exists(full) else print('  (không có file)')

## 2. Unit test

Chạy toàn bộ `test/runtests.jl`: bộ đọc file, so với đáp án vét cạn, mọi tổ hợp bật/tắt tối ưu, so từng itemset với output SPMF đã lưu trong `test/fixtures/spmf/` (không cần Java), recovery và luật kết hợp. Dữ liệu ngẫu nhiên trong test dùng seed cố định. Testset "data/benchmark (nếu có)" chạy thêm `fpclose` ở minsup 90% trên các dataset vừa tải (nếu mục 1b lỗi thì testset này tự bỏ qua). Ô dưới chỉ in 30 dòng cuối, gồm bảng **Test Summary**.

In [ ]:
%%time
!julia --project=. test/runtests.jl 2>&1 | tail -n 30

## 3. Demo FPClose (CLI `demo.jl`)

CSDL `data/toy/pasquier99.txt` (5 giao dịch, ví dụ chính thức của SPMF), minsup 40%. Lệnh ghi ba file: tập phổ biến đóng, mọi tập phổ biến (bước recovery) và 10 luật kết hợp có lift cao nhất với confidence ≥ 0,6. Thời gian từng giai đoạn được in ra stderr. Sau đó chạy đúng dạng lệnh `demo.jl --minsup <tỉ lệ> data/benchmark/<file>.txt` trên một dataset benchmark thật (chess, minsup 0,8).

In [ ]:
!mkdir -p out
!echo '== Input: data/toy/pasquier99.txt' && cat data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 40% --output-closed out/demo_closed.txt --output-all out/demo_all.txt --rules out/demo_rules.txt --minconf 0.6 --top 10 data/toy/pasquier99.txt

In [ ]:
import os
for title, path in [('Tập phổ biến đóng', 'out/demo_closed.txt'),
                    ('Mọi tập phổ biến', 'out/demo_all.txt'),
                    ('Luật kết hợp top-10 theo lift', 'out/demo_rules.txt')]:
    print('=' * 10, title, f'({path})')
    print(open(path, encoding='utf-8').read() if os.path.exists(path) else '(không có file)')

In [ ]:
%%time
# Đúng dạng ví dụ của đề: julia --project=. demo.jl --minsup <tỉ lệ> data/benchmark/<file>.txt
# Không có tham số ghi file nên tập đóng in ra stdout; chuyển stdout vào file để chỉ xem vài dòng đầu.
# Số tập đóng và thời gian từng giai đoạn vẫn hiện trong ô (stderr).
if ensure_data(['chess']):
    os.makedirs(os.path.join(REPO_DIR, 'out'), exist_ok=True)
    sh('julia --project=. demo.jl --minsup 0.8 data/benchmark/chess.txt > out/chess_0.8_closed.txt')
    out_path = os.path.join(REPO_DIR, 'out', 'chess_0.8_closed.txt')
    if os.path.exists(out_path):
        lines = open(out_path, encoding='utf-8').read().splitlines()
        print('Số tập phổ biến đóng (chess, minsup 0,8):', len(lines))
        print('\n'.join(lines[:5]))

## 4. Đối chiếu với SPMF

- Dữ liệu: dùng chess và mushrooms của mục 1b (thiếu thì tự gọi `data_setup.jl --only chess,mushrooms`).
- `scripts/reference_check.jl`: trên Linux tự tải Java 25 (Temurin) và `spmf.jar` về `tools/`, rồi kiểm số closed itemset của SPMF với số tham chiếu. Kết quả ghi vào `results/reference/spmf_check_chess+mushrooms.csv`, là file riêng, không ghi đè `spmf_check.csv` đã commit. Cờ `--skip-fimi` bỏ bước biên dịch FIMI để tiết kiệm thời gian.
- `ensure_java()` thêm JDK 25 vào PATH (và tự tải lại nếu bước trên lỗi), vì `compare_spmf.jl` gọi `java` trên PATH.
- `scripts/compare_spmf.jl`: so TỪNG itemset và support của cài đặt nhóm với SPMF, trên mọi CSDL toy và một cấu hình của chess.

In [ ]:
ensure_data(['chess', 'mushrooms'])

In [ ]:
%%time
!julia --project=. scripts/reference_check.jl --datasets chess,mushrooms --skip-fimi

In [ ]:
# compare_spmf.jl và run_benchmark_grid.jl gọi `java` trên PATH: trỏ PATH tới JDK 25 trong tools/ (tự tải nếu chưa có).
ensure_java()

In [ ]:
need_py('pandas')
import os, pandas as pd
p = 'results/reference/spmf_check_chess+mushrooms.csv'
display(pd.read_csv(p)) if os.path.exists(p) else print('Chưa có', p)

In [ ]:
%%time
# Bảng khớp từng itemset trên mọi CSDL toy (closed và mọi FI), sau đó một cấu hình của chess (80%)
!julia --project=. scripts/compare_spmf.jl --table
!julia --project=. scripts/compare_spmf.jl data/benchmark/chess.txt 80%

## 5. Thực nghiệm

### 5a. Lưới rút gọn chạy trên Colab (chỉ để minh họa)

`run_benchmark_grid.jl --quick --datasets chess --only correctness,runtime`: chỉ chess, 2–3 mức minsup, 1 lần đo, để dành thời gian cho Chương 3. Mọi file kết quả có hậu tố `_quick` (kể cả `results/ENVIRONMENT_quick.md`) nên **không ghi đè** kết quả đã commit; ô cuối của mục này kiểm tra lại bằng `git status`. Chạy lại ô thì các cấu hình đã đo được bỏ qua.

**Số đo trên Colab chỉ để minh họa. Số chính thức là bản đo trên máy MSI ở mục 5b.**

In [ ]:
%%time
if ensure_data(['chess']):
    ensure_java()
    sh(f'julia --project=. scripts/run_benchmark_grid.jl --quick --datasets chess --only correctness,runtime --machine {MACHINE_LABEL}')

In [ ]:
need_py('pandas')
import os, pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
for p in ['results/correctness/grid_correctness_quick.csv',
          'results/runtime/grid_runtime_summary_quick.csv']:
    print('==', p)
    display(pd.read_csv(p)) if os.path.exists(p) else print('  (chưa có file)')

In [ ]:
# Chỉ được có file MỚI (dòng '??', đuôi _quick hoặc spmf_check_chess+mushrooms); không có dòng ' M' (file đã commit bị sửa).
!git status --short results/

### 5b. Kết quả đo đầy đủ (máy MSI, đã commit)

Số liệu chính thức của Chương 2 được đo trên máy MSI; cấu hình máy ở `results/ENVIRONMENT.md`. Bảng tổng hợp sinh bằng `scripts/summarize_results.jl`, hình vẽ bằng `scripts/plot_results.jl`. Thời gian là trung vị 3 lần đo. Khi so với SPMF, thời gian của cả hai bên đều tính cả bước ghi file. Bộ nhớ của SPMF là heap JVM nên không so trực tiếp với bộ nhớ của nhóm.

In [ ]:
print(open('results/ENVIRONMENT.md', encoding='utf-8').read())

In [ ]:
need_py('pandas')
import os, pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
SUMMARY = 'results/summary'

def show(name, title, cols=None):
    path = f'{SUMMARY}/{name}.csv'
    if not os.path.exists(path):
        print('Chưa có', path)
        return
    df = pd.read_csv(path)
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    print(f'== {title} ({path}, {len(df)} dòng)')
    display(df)

show('correctness_summary', 'Tính đúng đắn: nhóm vs SPMF')
show('speedup', 'Thời gian ours vs SPMF (SPMF / ours > 1 nghĩa là ours nhanh hơn)')
show('time_breakdown', 'Phân rã thời gian của ours')
show('ablation', 'Ablation các tối ưu (% so với optimized)',
     cols=['dataset', 'minsup_pct', 'variant', 'n_closed', 'mine_s', 'mine_s_vs_opt_pct',
           'mem_delta_mb', 'mem_delta_mb_vs_opt_pct', 'alloc_mb', 'alloc_mb_vs_opt_pct'])
show('scalability', 'Khả năng mở rộng (Retail 10%–100%)')
show('txlen', 'Ảnh hưởng độ dài giao dịch')

In [ ]:
import os
from IPython.display import Image, Markdown, display
FIG = 'results/figures'
figures = [
    ('runtime_grid', 'Thời gian chạy theo minsup, mọi dataset (ours vs SPMF)', 900),
    ('itemsets_vs_minsup', 'Số tập phổ biến đóng theo minsup', 900),
    ('time_breakdown', 'Phân rã thời gian của ours', 600),
    ('memory_variants_grid', 'Bộ nhớ thực dùng theo biến thể', 900),
    ('time_variants_grid', 'Thời gian khai thác theo biến thể', 900),
    ('ablation_time_relative', 'Ablation thời gian (so với optimized)', 600),
    ('scalability_time', 'Khả năng mở rộng theo kích thước', 600),
    ('txlen_time', 'Thời gian theo độ dài giao dịch', 600),
    ('txlen_itemsets', 'Số tập đóng theo độ dài giao dịch', 600),
]
for name, caption, width in figures:
    path = f'{FIG}/{name}.png'
    if os.path.exists(path):
        display(Markdown(f'**{caption}** (`{path}`)'))
        display(Image(filename=path, width=width))
    else:
        print('Chưa có', path)

## 6. Ứng dụng (Chương 3)

Hai ứng dụng được làm ngay trong notebook này. **Ràng buộc chung:** mọi itemset và luật kết hợp phải sinh bằng chính FPClose của nhóm, qua CLI `demo.jl` hoặc các hàm trong `src/` (`read_spmf`, `fpclose`, `recover_all_frequent`, `generate_rules`, ...). **KHÔNG** dùng mlxtend hay bất kỳ thư viện khai thác tập phổ biến (FIM) nào khác. Thư viện xử lý ảnh, k-means, pandas chỉ được dùng ở bước tạo giao dịch và trình bày kết quả.

### 6.1 Ứng dụng 1 – Market basket

*(Placeholder — sẽ bổ sung các ô code bên dưới.)*

- **Mục tiêu:** tìm các nhóm sản phẩm thường được mua cùng nhau và luật kết hợp (support, confidence, lift) để gợi ý bán chéo / sắp xếp kệ hàng.
- **Dữ liệu:** TODO — tên bộ dữ liệu giỏ hàng, nguồn, giấy phép, kích thước; cách tải nằm trong notebook.
- **Biến đổi dữ liệu thành giao dịch (đặt trong notebook):** gom các dòng theo mã hóa đơn thành tập mã sản phẩm; bỏ hóa đơn hủy, số lượng âm, mã không phải sản phẩm; ánh xạ mỗi sản phẩm sang một số nguyên ≥ 0 và lưu bảng ánh xạ để dịch ngược; ghi file định dạng SPMF (mỗi dòng một giao dịch, item cách nhau dấu cách).
- **Các bước:**
  1. Thống kê dữ liệu giao dịch (số giao dịch, số item, độ dài trung bình).
  2. Chọn minsup và minconf, có giải thích.
  3. Chạy FPClose của nhóm: `demo.jl --minsup ... --output-closed ... --rules ... --minconf ... --top k`.
  4. Dịch item về tên sản phẩm; trình bày các tập đóng tiêu biểu và các luật có lift cao nhất.
  5. Nhận xét ý nghĩa kinh doanh và hạn chế.

### 6.2 Ứng dụng 2 – Visual patterns (BoVW)

*(Placeholder — sẽ bổ sung các ô code bên dưới.)*

- **Mục tiêu:** tìm các tổ hợp "visual word" thường cùng xuất hiện trong ảnh của từng lớp, tức các mẫu thị giác đặc trưng của lớp đó (Bag of Visual Words).
- **Dữ liệu:** TODO — bộ ảnh nhỏ có nhãn lớp (tên, nguồn, giấy phép, số ảnh dùng); cách tải nằm trong notebook.
- **Biến đổi dữ liệu thành giao dịch (đặt trong notebook):** trích đặc trưng cục bộ của từng ảnh (vd ORB/SIFT bằng OpenCV); gom cụm toàn bộ descriptor bằng k-means (vd scikit-learn) để lập từ điển k visual word; mỗi ảnh thành một giao dịch là tập id các visual word xuất hiện trong ảnh (có thể lọc word quá phổ biến); ghi file định dạng SPMF, mỗi lớp một file. OpenCV và k-means chỉ dùng ở bước này.
- **Các bước:**
  1. Chọn k và cách lọc word, có giải thích; thống kê giao dịch thu được.
  2. Chạy FPClose của nhóm trên giao dịch của từng lớp.
  3. So sánh các tập đóng giữa các lớp, tìm tập đặc trưng riêng của mỗi lớp.
  4. Trực quan hóa: vẽ lại vị trí các keypoint thuộc một tập đóng lên vài ảnh mẫu.
  5. Nhận xét và hạn chế.

## 7. Kết luận

TODO: tóm tắt kết quả (tính đúng đắn so với SPMF, so sánh thời gian, ảnh hưởng của các tối ưu) và hướng phát triển.

In [ ]:
print(f'Tổng thời gian chạy notebook: {(time.time() - T_START) / 60:.1f} phút')